# Sesión 3 · Fundamentos de IA Generativa y Genie
**Databricks AI Engineer** — caso Neptuno

Hoy dejamos de tratar al modelo como una caja negra. Vamos a medir cómo cambian sus respuestas
cuando recibe contexto, un contrato de salida y metadatos confiables.

Al terminar tendrás un laboratorio de prompts trazable y una tabla lista para comparar respuestas.

## 0 · Conectar con tu catálogo

In [0]:
# Al importar o actualizar el notebook, Databricks no ejecuta el código automáticamente.
# Esta debe ser la primera celda que corras. Escribe arriba el nombre completo del catálogo
# de S01 y S02, por ejemplo: neptuno_tunombre.
dbutils.widgets.text("catalogo", "", "Tu catálogo de S01–S02")
print("✅ Widget creado. Escribe arriba el nombre completo de tu catálogo y vuelve a ejecutar esta celda.")

✅ Widget creado. Escribe arriba el nombre completo de tu catálogo y vuelve a ejecutar esta celda.


In [0]:
# `re` es la librería estándar de Python para expresiones regulares; aquí valida el formato del catálogo.
import re
# Los widgets permiten cambiar catálogo y endpoint desde la interfaz, sin editar el código.
#dbutils.widgets.text("modelo", "system.ai.gpt-5-6-luna", "Endpoint de Foundation Model") No esta en Free Tier
dbutils.widgets.text("modelo", "databricks-gpt-oss-20b", "Endpoint de Foundation Model")
CATALOGO = dbutils.widgets.get("catalogo").strip().lower()
MODELO = dbutils.widgets.get("modelo").strip()
assert re.fullmatch(r"neptuno_[a-z0-9_]+", CATALOGO or ""), "Usa tu catálogo neptuno_<nombre>."
assert CATALOGO in {r.catalog.lower() for r in spark.sql("SHOW CATALOGS").collect()}, f"No existe {CATALOGO}."
assert MODELO, "Indica un endpoint disponible en tu workspace."
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.ai_lab")
print(f"✅ catálogo={CATALOGO} · modelo={MODELO}")

✅ catálogo=neptuno_emerson_suarez · modelo=databricks-gpt-oss-20b


### Troubleshooting: ¿tu workspace tiene este modelo?
Cada workspace de Databricks tiene disponibles distintos modelos según cuándo se creó y su
región — no es lo mismo para todos. Si el `ai_query` de más abajo falla con
`RESOURCE_DOES_NOT_EXIST`, corré esta celda antes de pedir ayuda: te dice exactamente qué
endpoints legacy (`databricks-...`) tiene TU workspace. Los modelos `system.ai.*` (como el
default de este notebook) no aparecen en esta lista aunque sí funcionen — es un catálogo
distinto (Unity AI Gateway), no un bug de la celda.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoints_activos = [ep.name for ep in w.serving_endpoints.list()]

if MODELO.startswith("system.ai."):
    print(f"ℹ️  '{MODELO}' es un modelo de Unity AI Gateway: no aparece en esta lista aunque esté disponible.")
    print("   Si igual falla, revisá en el menú AI/ML → AI Gateway → Models que aparezca listado ahí.")
elif MODELO in endpoints_activos:
    print(f"✅ El endpoint '{MODELO}' existe en este workspace.")
else:
    print(f"❌ El endpoint '{MODELO}' NO existe en este workspace.")
    print("Endpoints disponibles:", endpoints_activos)
    print("Sugerencia: cambiá el widget 'modelo' a 'system.ai.gpt-5-6-luna'.")

✅ El endpoint 'databricks-gpt-oss-20b' existe en este workspace.


## 1 · Un modelo predice texto; no consulta la verdad
Temperatura baja reduce variación, pero no agrega conocimiento. Comparamos una pregunta que
Neptuno **no puede responder** porque no existe costo de mercadería.

In [0]:
preguntas = [
    ("margen", "¿Cuál es el margen de la categoría Bebidas?"),
    ("venta", "Explica en una frase qué significa venta neta."),
]
spark.createDataFrame(preguntas, "id string, pregunta string").createOrReplaceTempView("preguntas_s03")

### Llamada controlada con `ai_query`
Si el endpoint sugerido no existe, elige uno habilitado en **Serving → Foundation Model APIs**
y cambia el widget `modelo`.

In [0]:
respuestas_base = spark.sql(f"""
SELECT id, pregunta,
       ai_query('{MODELO}', concat(
         'Responde en español, en máximo 60 palabras. Pregunta: ', pregunta
       )) AS respuesta
FROM preguntas_s03
""")
display(respuestas_base)
assert respuestas_base.count() == 2

id,pregunta,respuesta
margen,¿Cuál es el margen de la categoría Bebidas?,"Lo siento, no dispongo de la información actual sobre el margen de la categoría Bebidas."
venta,Explica en una frase qué significa venta neta.,"La venta neta es el total de ingresos por ventas después de deducir devoluciones, descuentos y bonificaciones, reflejando el ingreso real que la empresa recibe por sus productos o servicios."


### Parámetros reales del endpoint
`ai_query` permite pasar parámetros mediante `modelParameters`, pero el endpoint puede
rechazar algunos. En este workspace verificamos que `max_tokens` funciona; `temperature` y
`top_p` no están soportados por `system.ai.gpt-5-6-luna`, así que no los simulamos en clase.

In [0]:
respuesta_limitada = spark.sql(f"""
SELECT ai_query(
  '{MODELO}',
  'Responde únicamente OK.',
  -- `modelParameters` envía opciones específicas al endpoint.
  -- `named_struct` crea el objeto de parámetros que espera `ai_query`.
  modelParameters => named_struct('max_tokens', 100), -- luna cambiar a 20
  failOnError => false
) AS respuesta
""").first()["respuesta"]
print(respuesta_limitada)
assert respuesta_limitada["errorMessage"] is None, respuesta_limitada
assert respuesta_limitada["result"].strip() == "OK"
print("✅ max_tokens aplicado y verificado en el endpoint seleccionado.")

Row(result='OK', errorMessage=None)
✅ max_tokens aplicado y verificado en el endpoint seleccionado.


## 2 · Prompt = tarea + contexto + límites + formato
El contexto no debe decirle al modelo qué inventar: debe decirle qué evidencia existe y qué
hacer cuando esa evidencia no alcanza.

In [0]:
contexto = f"""
Eres analista de Neptuno. La tabla {CATALOGO}.gold.ventas_por_categoria_mes contiene ventas netas.
La venta neta ya descuenta promociones. El modelo de datos NO contiene costo de mercadería.
Por eso NO permite calcular margen, utilidad ni rentabilidad. Si falta evidencia, dilo explícitamente.
""".strip()

prompt_seguro = contexto + "\nPregunta: ¿Cuál es el margen de la categoría Bebidas?\n" + (
    "Devuelve exactamente dos campos: respuesta y evidencia_usada. Máximo 80 palabras."
)
segura = spark.sql(f"SELECT ai_query('{MODELO}', {repr(prompt_seguro)}) AS respuesta").first()["respuesta"]
print(segura)
assert len(segura.strip()) > 10

{
  "respuesta": "No se puede calcular el margen de la categoría Bebidas porque no se dispone de datos de costo de mercadería.",
  "evidencia_usada": "La tabla neptuno_emerson_suarez.gold.ventas_por_categoria_mes contiene ventas netas, pero no incluye costos de mercadería."
}


## 3 · Grounding: darle hechos, no toda la tabla
Recuperamos primero un agregado verificable y recién después pedimos lenguaje natural.

In [0]:
tabla_ventas = f"{CATALOGO}.gold.ventas_por_categoria_mes"
assert spark.catalog.tableExists(tabla_ventas), f"Falta {tabla_ventas}; termina la S02."

hechos = spark.sql(f"""
SELECT categoria, ROUND(SUM(ingreso_neto), 2) AS venta_neta
FROM {tabla_ventas}
GROUP BY categoria
ORDER BY venta_neta DESC
LIMIT 5
""").toPandas().to_dict("records")  # Pasamos el agregado a Python para incrustarlo en el prompt.
print(hechos)
assert hechos, "La tabla Gold no devolvió hechos."

[{'categoria': 'Bebidas', 'venta_neta': 267868.2}, {'categoria': 'Lacteos', 'venta_neta': 234507.32}, {'categoria': 'Reposteria', 'venta_neta': 167357.25}, {'categoria': 'Carnes y Aves', 'venta_neta': 163022.38}, {'categoria': 'Pescados y Mariscos', 'venta_neta': 131261.75}]


In [0]:
prompt_grounded = f"""
Eres analista de Neptuno. Usa EXCLUSIVAMENTE estos hechos: {hechos}
Resume el top de categorías en tres viñetas. No calcules margen: no hay costos.
Incluye los números utilizados y termina con: Fuente: {tabla_ventas}
""".strip()
respuesta_grounded = spark.sql(
    f"SELECT ai_query('{MODELO}', {repr(prompt_grounded)}) AS respuesta"
).first()["respuesta"]
print(respuesta_grounded)
assert tabla_ventas in respuesta_grounded, "La respuesta no incluyó la fuente solicitada."

- Bebidas: 267 868,20  
- Lácteos: 234 507,32  
- Repostería: 167 357,25  

Fuente: neptuno_emerson_suarez.gold.ventas_por_categoria_mes


## 4 · Persistir el experimento
Una demo se mira; un experimento se guarda con sus entradas y resultados.

In [0]:
# UTC evita mezclar zonas horarias al comparar experimentos.
from datetime import datetime, timezone

filas = [
    ("sin_contexto", "¿Cuál es el margen de Bebidas?", respuestas_base.filter("id='margen'").first()["respuesta"]),
    ("con_limite", "¿Cuál es el margen de Bebidas?", segura),
    ("grounded", "Resume el top de categorías", respuesta_grounded),
]
df_eval = (spark.createDataFrame(filas, "variante string, pregunta string, respuesta string")
                 .withColumn("registrado_ts", __import__("pyspark").sql.functions.lit(datetime.now(timezone.utc))))
df_eval.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(  # Persistimos el experimento en Delta.
    f"{CATALOGO}.ai_lab.experimentos_prompt"
)

## 5 · De prompt a Genie
En Genie el contexto se gobierna con cuatro capas: tablas autorizadas, comentarios de Unity
Catalog, instrucciones del space y SQL de confianza. Prueba en el space curado:

1. “¿Cuál es el margen de Bebidas?”
2. “¿Cuánto vendimos de Bebidas en 2025?”
3. Inspecciona el SQL y la evidencia.

**Criterio:** una buena respuesta no es la que siempre entrega un número; es la que sabe cuándo
la plataforma no contiene evidencia suficiente.

## 6 · Verificación y entregable

In [0]:
guardado = spark.table(f"{CATALOGO}.ai_lab.experimentos_prompt")
assert guardado.count() == 3
assert set(r.variante for r in guardado.select("variante").collect()) == {"sin_contexto", "con_limite", "grounded"}
display(guardado)
print("✅ Entregable S03: 3 variantes guardadas y comparables.")

variante,pregunta,respuesta,registrado_ts
sin_contexto,¿Cuál es el margen de Bebidas?,"El margen de la categoría Bebidas es del 20 % al 30 %, variando según el subproducto.",2026-09-09T03:07:54.934Z
con_limite,¿Cuál es el margen de Bebidas?,"**respuesta**: No se puede determinar el margen de la categoría Bebidas porque no se dispone de datos de costo de mercadería. **evidencia_usada**: La tabla neptuno_emerson_suarez.gold.ventas_por_categoria_mes solo contiene ventas netas, sin costos.",2026-09-09T03:07:54.934Z
grounded,Resume el top de categorías,"- **Bebidas**: 267 868,20 - **Lácteos**: 234 507,32 - **Repostería**: 167 357,25 Fuente: neptuno_emerson_suarez.gold.ventas_por_categoria_mes",2026-09-09T03:07:54.934Z


✅ Entregable S03: 3 variantes guardadas y comparables.


### Reto
Diseña una cuarta variante para “¿qué productos son más rentables?”. Debe negarse a inventar
rentabilidad, explicar qué dato falta y ofrecer una pregunta alternativa que sí pueda responder.

### ¿Qué productos son más rentables?

In [0]:
pregunta_reto = "¿Qué productos son más rentables?"

prompt_reto = f"""
Eres analista de Neptuno. Responde exclusivamente con base en el alcance documentado.
La plataforma contiene ventas netas y unidades vendidas, pero NO contiene costos de
mercadería por producto. Sin costos no se puede calcular utilidad, margen ni rentabilidad.

Pregunta: {pregunta_reto}

Si la pregunta no puede responderse, debes:
1. negarte explícitamente a inventar un ranking;
2. indicar exactamente qué dato falta;
3. proponer como alternativa una pregunta sobre productos con mayor venta neta.
Máximo 100 palabras.
""".strip()

respuesta_reto = spark.sql(
    f"SELECT ai_query('{MODELO}', {repr(prompt_reto)}) AS respuesta"
).first()["respuesta"]

print(respuesta_reto)

texto_reto = respuesta_reto.lower()
assert "costo" in texto_reto, "La respuesta no explicó el dato ausente"
assert any(palabra in texto_reto for palabra in ["no puedo", "no es posible", "no se puede"]), (
    "La respuesta no se abstuvo explícitamente"
)

No puedo generar un ranking de rentabilidad porque la base de datos no incluye el costo de mercadería por producto, necesario para calcular utilidad, margen y rentabilidad.  

Alternativa: ¿Cuáles son los productos con mayor venta neta?


In [0]:
from datetime import datetime, timezone

filas_finales = [
    (
        "sin_contexto",
        "¿Cuál es el margen de Bebidas?",
        respuestas_base.filter("id='margen'").first()["respuesta"],
    ),
    (
        "con_limite",
        "¿Cuál es el margen de Bebidas?",
        segura,
    ),
    (
        "grounded",
        "Resume el top de categorías",
        respuesta_grounded,
    ),
    (
        "reto_rentabilidad",
        pregunta_reto,
        respuesta_reto,
    ),
]

df_final = (
    spark.createDataFrame(
        filas_finales,
        "variante string, pregunta string, respuesta string",
    )
    .withColumn(
        "registrado_ts",
        __import__("pyspark").sql.functions.lit(datetime.now(timezone.utc)),
    )
)

df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOGO}.ai_lab.experimentos_prompt"
)

print("✅ Tabla final guardada con cuatro variantes")

✅ Tabla final guardada con cuatro variantes


In [0]:
tabla_experimentos = f"{CATALOGO}.ai_lab.experimentos_prompt"

assert spark.catalog.tableExists(tabla_experimentos), f"Falta {tabla_experimentos}"

final = spark.table(tabla_experimentos)
variantes_esperadas = {
    "sin_contexto",
    "con_limite",
    "grounded",
    "reto_rentabilidad",
}

assert final.count() == 4, f"Se esperaban 4 filas y existen {final.count()}"
assert set(r.variante for r in final.select("variante").collect()) == variantes_esperadas
assert final.filter("respuesta IS NULL OR TRIM(respuesta) = ''").count() == 0
assert final.filter("registrado_ts IS NULL").count() == 0

display(final.orderBy("variante"))
print("🎉 Sesión 3 completa: cuatro variantes persistidas y verificadas")

variante,pregunta,respuesta,registrado_ts
con_limite,¿Cuál es el margen de Bebidas?,"{ ""respuesta"": ""No se puede calcular el margen de la categoría Bebidas porque no se dispone de datos de costo de mercadería."", ""evidencia_usada"": ""La tabla neptuno_emerson_suarez.gold.ventas_por_categoria_mes contiene ventas netas, pero no incluye costos de mercadería."" }",2026-09-09T03:53:16.346Z
grounded,Resume el top de categorías,"- Bebidas: 267 868,20 - Lácteos: 234 507,32 - Repostería: 167 357,25 Fuente: neptuno_emerson_suarez.gold.ventas_por_categoria_mes",2026-09-09T03:53:16.346Z
reto_rentabilidad,¿Qué productos son más rentables?,"No puedo generar un ranking de rentabilidad porque la base de datos no incluye el costo de mercadería por producto, necesario para calcular utilidad, margen y rentabilidad. Alternativa: ¿Cuáles son los productos con mayor venta neta?",2026-09-09T03:53:16.346Z
sin_contexto,¿Cuál es el margen de Bebidas?,"El margen de la categoría Bebidas es del 20 % al 30 %, variando según el subproducto.",2026-09-09T03:53:16.346Z


🎉 Sesión 3 completa: cuatro variantes persistidas y verificadas
